# Laboratorio L7 - Classificare con regole scritte a mano

B.6 - Data science e Machine Learning: dai dati ai modelli

File necessari: `pinguini.csv`, `L7_schede_pinguini.csv`.

Dati: Palmer Penguins, Horst AM, Hill AP, Gorman KB (2020), licenza CC0, https://allisonhorst.github.io/palmerpenguins/

## 0. Preparazione

`dropna(subset=[...])` tiene solo i pinguini con tutte le misure usate nel laboratorio.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def controlla(condizione, suggerimento="Non ancora: rileggere la consegna."):
    """Stampa un riscontro immediato sull'esercizio."""
    print("Corretto." if condizione else suggerimento)

pinguini = pd.read_csv("pinguini.csv").dropna(subset=["becco_lunghezza_mm", "pinna_lunghezza_mm", "massa_g"])
SPECIE = ["Adelie", "Chinstrap", "Gentoo"]
COLORI = {"Adelie": "#2a78d6", "Chinstrap": "#eb6834", "Gentoo": "#1baf7a"}
CHIARI = ["#cfe0f5", "#fad6c6", "#c4ecdd"]
MARCATORI = {"Adelie": "o", "Chinstrap": "s", "Gentoo": "^"}
print(len(pinguini), "pinguini con tutte le misure")

## 1. Le regole del gruppo (esercizio 1)

Trascrivere nella funzione `classifica` le regole scritte dal gruppo sulle schede.

- `def classifica(becco, pinna):` definisce una funzione con due parametri, la lunghezza del becco e della pinna
- `if condizione:` esegue il blocco indentato solo se la condizione è vera; `elif` (altrimenti se) controlla un'altra condizione; `else` (altrimenti) si applica in tutti gli altri casi
- `return` restituisce il risultato della funzione
- le regole vengono controllate nell'ordine: la prima vera decide

Le soglie scritte qui sotto sono un esempio: sostituirle con quelle del gruppo.

In [ ]:
def classifica(becco, pinna):
    if pinna > 206:
        return "Gentoo"
    elif becco > 43:
        return "Chinstrap"
    else:
        return "Adelie"

print(classifica(39.1, 181))   # un pinguino con becco corto e pinna corta
print(classifica(50.0, 220))

## 2. Verifica sulle 24 schede (esercizio 2)

- `zip(a, b)` scorre insieme due colonne, una coppia di valori alla volta
- `[classifica(b, p) for b, p in ...]`: applica la funzione a ogni coppia e raccoglie i risultati in una lista
- `(previsti == veri).mean()`: frazione di previsioni corrette, cioè l'**accuratezza** (`True` vale 1, `False` vale 0)

In [ ]:
schede = pd.read_csv("L7_schede_pinguini.csv")
schede["prevista"] = [classifica(b, p) for b, p in zip(schede["becco_lunghezza_mm"], schede["pinna_lunghezza_mm"])]
schede["corretta"] = schede["prevista"] == schede["specie"]
print("accuratezza sulle schede:", round(schede["corretta"].mean(), 3))
schede[~schede["corretta"]]      # ~ inverte: le schede classificate male

## 3. Verifica su tutti i pinguini (esercizio 3)

Le regole sono state scritte guardando 24 pinguini. Funzionano anche sugli altri?

In [ ]:
previsti = [classifica(b, p) for b, p in zip(pinguini["becco_lunghezza_mm"], pinguini["pinna_lunghezza_mm"])]
accuratezza = (pinguini["specie"] == previsti).mean()
print("accuratezza su", len(pinguini), "pinguini:", round(accuratezza, 3))

Tabella degli errori: righe = specie vera, colonne = specie prevista. I numeri fuori dalla diagonale sono gli errori.

In [ ]:
pd.crosstab(pinguini["specie"], pd.Series(previsti, index=pinguini.index, name="prevista"))

Completare: assegnare a `n_errori` il numero di pinguini classificati male (suggerimento: `(pinguini["specie"] != previsti).sum()`).

In [ ]:
n_errori = ___
print(n_errori)
controlla(n_errori == round(len(pinguini) * (1 - accuratezza)))

## 4. Il confine di decisione (esercizio 4)

La funzione `disegna_confine` colora ogni punto del piano con la specie che le regole assegnerebbero a un pinguino con quelle misure, e disegna sopra i pinguini veri. Non è necessario capirne il codice: è uno strumento per vedere le regole.

In [ ]:
from matplotlib.colors import ListedColormap

def disegna_confine(prevedi, X, y, titolo=""):
    """Colora ogni punto del piano con la specie prevista e disegna sopra i pinguini veri.
    prevedi: funzione che riceve una tabella di punti (due colonne) e restituisce le specie previste."""
    a, b = X.iloc[:, 0], X.iloc[:, 1]
    ga, gb = np.meshgrid(np.linspace(a.min() - 1, a.max() + 1, 250),
                         np.linspace(b.min() - 2, b.max() + 2, 250))
    griglia = np.c_[ga.ravel(), gb.ravel()]                     # 62500 punti del piano
    previsti = np.array(prevedi(griglia))
    indici = np.array([SPECIE.index(s) for s in previsti]).reshape(ga.shape)
    plt.contourf(ga, gb, indici, levels=[-0.5, 0.5, 1.5, 2.5], cmap=ListedColormap(CHIARI))
    for s in SPECIE:
        sel = (y == s).to_numpy()
        plt.scatter(a[sel], b[sel], color=COLORI[s], marker=MARCATORI[s], s=18,
                    edgecolor="white", linewidth=0.5, label=s)
    plt.xlabel(X.columns[0]); plt.ylabel(X.columns[1]); plt.title(titolo)
    plt.legend(loc="lower right")
    plt.show()

In [ ]:
X = pinguini[["becco_lunghezza_mm", "pinna_lunghezza_mm"]]
y = pinguini["specie"]

def prevedi_con_regole(punti):
    return [classifica(b, p) for b, p in punti]

disegna_confine(prevedi_con_regole, X, y, "Confine delle regole scritte a mano")

**Domande**

- Che forma hanno le zone? Perché le regole a soglia producono confini con linee orizzontali e verticali?
- Dove si concentrano gli errori?
- Modificare le soglie per ridurre gli errori. Quale accuratezza massima si riesce a ottenere? Diventa più difficile migliorare, a un certo punto?

Risposte:

## 5. Approfondimento (esercizio 5)

Scrivere una seconda funzione `classifica2` che usa anche la profondità del becco (terzo parametro) e confrontarne l'accuratezza con `classifica`.

In [ ]:
def classifica2(becco, profondita, pinna):
    if profondita < 16.5:
        return "Gentoo"
    elif becco > 44:
        return "Chinstrap"
    else:
        return "Adelie"

prev2 = [classifica2(b, d, p) for b, d, p in zip(pinguini["becco_lunghezza_mm"], pinguini["becco_profondita_mm"], pinguini["pinna_lunghezza_mm"])]
print("accuratezza con la profondità:", round((pinguini["specie"] == prev2).mean(), 3))